## Updated Project Direction: From Profiling to Cleaning

Our business objective is to use the data available to decide which movie attributes (genre, budget range, keywords, director and cast) to prioritize for Netflix’s next year of original films within a $10B budget. 

Profiling validated that the datasets contain the information required to conduct analysis. There are thousands of movies with ratings, revenue, and budget data across multiple genres.

Profiling revealed that about 50% of the movies are missing revenue or budget data, which led us to decide on 2 different success metrics. One success metric based on roi and another based on movie rating.

Our output ABT, movies_abt, will allow us to create the following models:

***Model 1 - used if budget and revenue data is available:***

***Movie Success (ExpROI)*** = *movies[genre_list] + keywords[keywords_list] + movies[budget] + avg(ratings[rating] by director) + avg(ratings[rating] by actor)*  

***ROI*** = *(movies[revenue] - movies[budget]) / movies[budget]*

A movie is assumed to be breakeven if revenue is double the budget. Our model will estimate ROI based on movie attributes.

***Model 2 - used if budget and revenue data is not available:***

 ***Exp Movie Rating avg(ratings[rating])*** = *movies[genre_list] + keywords[keywords_list] + avg(ratings[rating] by director) + avg(ratings[rating] by actor)*

 A highly rated movie will have a rating in the top 25% of movies.

These model specifications are preliminary. Final predictors, method and validation approach will be confirmed in Milestone 3, based on the completed ABT.

## ABT Preperation

At this point in our analysis we have stated a business decision, decided on a model or output, inspected the data, and created a cleaning plan. 

Throughout this section we will...
1) Create copies of the raw data so the original data remains available for comparison.
2) Create cleaning_log to log all changes.
3) Convert all columns to the appropriate data type format.
4) Aggregate ratings to make row unit level equal to 1 movie.
5) Create a ratings success threshold flag.
6) Convert multi-valued data into long data.
7) Drop columns that aren't needed.
8) Create derived column roi and roi success flag.
9) Filter out rows where mandatory fields are missing.
10) Merge tables to create the movies_abt.
11) Create derived column called release_month for further analysis.
12) Review the cleaning_log
13) Conduct final duplicate and missing values check.
14) Use SQL to validate row count of outputs.
15) Save created tables to outputs folder.
16) State how AI was utilized throughout the process.

After each step we will review changes made. By the end of this file we will have a preliminary ABT.

---
## Loading The Dataframe

The initial step in this project is to load the required libraries and the datasets. 
 
The libraries required are...   
        - pathlib : a file handler that treats paths as objects rather than text strings.   
        - pandas : a library of functions that are used to manipulate dataframes.  
        - numpy : a library of math functions.  

Next, we define the root folder where the project folders and files will be saved.  

The project directory contains the following folders...  
        - The data folder : This folder contains raw data files.  
        - The output folder : This folder contains the final ABT and processed data files.    
        - The report folder : This folder contains html reports, graphs, and analysis.  



In [1380]:
# This code loads libraries and sets directories for the project.
#  pathlib is a file handler. treats paths as objects not strings
# pandas brings in df functions
# numpy provides math functions.

#Checking my Python environment 
import sys; print(sys.executable); print(sys.version)

from pathlib import Path
import pandas as pd
import numpy as np

#Set repository root to the current working directory. One step above readme location.

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "README.md").exists() and (REPO_ROOT.parent / "README.md").exists():
    REPO_ROOT = REPO_ROOT.parent

display(REPO_ROOT)

#Create directories for data, outputs, and reports if they don't exist.
DATA_DIR = REPO_ROOT / "data"
RAW_DATA = REPO_ROOT / "data/raw"
PROCESSED_DATA = REPO_ROOT / "outputs/processed"
OUTPUT_DIR = REPO_ROOT / "outputs"
REPORTS_DIR = REPO_ROOT / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

#set max column display to 100 columns for pandas dataframes.
pd.set_option("display.max_columns", 100)

#This code loads the raw data into dataframes. And creates a column summary by table df.
# Load the datasets
movies = pd.read_csv(RAW_DATA / 'movies_metadata.csv')
ratings = pd.read_csv(RAW_DATA / 'ratings.csv', parse_dates=['rating_datetime'])
links = pd.read_csv(RAW_DATA / 'links.csv')
credits = pd.read_csv(RAW_DATA / 'credits_curated.csv')
keywords = pd.read_csv(RAW_DATA / 'keywords_curated.csv')

# Create a dictionary (keys paired with values) to hold the DataFrames for easier access. We can loop through tables, etc.
tables = {
    'movies': movies,
    'ratings': ratings,
    'links': links,
    'credits': credits,
    'keywords': keywords,
}

print("Datasets loaded into dataframes. Dictionary of dataframes called 'tables' created.")

c:\python314\python.exe
3.14.4 (tags/v3.14.4:23116f9, Apr  7 2026, 14:10:54) [MSC v.1944 64 bit (AMD64)]


WindowsPath('c:/Users/matth/Project-Option-1-Matt-Flores')

Datasets loaded into dataframes. Dictionary of dataframes called 'tables' created.


### Loading DuckDB

DuckDB is a small database engine that runs inside Python that allows us to run sql queries on dataframes and local files. We can use DuckDB to run final validations on our abt.

In [1381]:
import duckdb

def run_duckdb(query):
    """Run a DuckDB query when it has been filled in; keep blank practice cells runnable."""
    if not query or not query.strip():
        print("This practice cell is waiting for your SQL. Add your query inside the triple quotes, then run it again.")
        return pd.DataFrame()
    return duckdb.sql(query).df()

print("DuckDB version:", duckdb.__version__)

DuckDB version: 2.0.0


---
## Copying Raw Data 
Before we make any changes we must first create copies of the raw data. There are several reasons why creating copies of the raw data is important:
1) It preserves the original data for comparison purposes. 
2) It allows others to reproduce the analysis using the original data.
3) It let's us undo any changes made during cleanup and transformation.

In [1382]:
# Use .copy() to create working dataframes
movies_clean = movies.copy() 
ratings_clean = ratings.copy() 
links_clean = links.copy() 
credits_clean = credits.copy() 
keywords_clean = keywords.copy() 

print("All copies created!")

All copies created!


---
### Creating a cleaning log to record changes.

Before we start cleaning we will create an empty list to record the changes made throughout the cleaning and transformation process. After each change made to the data, we will record the method used, table and column impacted, a description of the change, reason, and notes.

In [1383]:
cleaning_log = []

print("cleaning log created")

cleaning log created


---
## Cleaning - Movies
### Convert realease_date from object to datetime
Converting from object to datetime allows us to use pandas functions designed for timeseries data. It also converts invalid dates to NaT (not a time).

In Milestone 1 we confirmed that there were no zero or missing release_date. There should be no NaT after conversion.

In [1384]:
# Dtype of release date before conversion.
release_date_type_before = movies_clean['release_date'].dtype

# isna count of release date before conversion.
release_date_missing_before = movies_clean['release_date'].isna().sum()

# Convert using pd.to_datetime
movies_clean['release_date'] = pd.to_datetime(movies_clean['release_date'],  errors="coerce")

# Dtype of release date after conversion.
release_date_type_after = movies_clean['release_date'].dtype

# Dtype of release date after conversion.
release_date_missing_after = movies_clean['release_date'].isna().sum()

# Print isna count summary.
print(f"release_date converted to NaT: {release_date_missing_after - release_date_missing_before}")

# Print type before and after
print(f"release_date converted from {release_date_type_before} to {release_date_type_after}")

print(f"release dates span from {movies_clean['release_date'].min().date()} and {movies_clean['release_date'].max().date()}")

# Append changes to cleaning log
cleaning_log.append({
    'method': 'conversion',
    'table': 'movies',
    'column': 'release_date',
    'description': 'object to datetime64 (errors=coerce)',
    'reason': 'enable date filtering and pandas date features',
    'notes': 'no NaT. min date is 1902-09-01, max date is 2016-09-23',
})

release_date converted to NaT: 0
release_date converted from str to datetime64[us]
release dates span from 1902-09-01 and 2016-09-23


All release_dates converted without errors. Movie release dates span from 1902 to 2016.

### Convert vote_count from float64 to Int64

vote_counts are whole numbers. Converting to Int64 will show them as whole numbers and reveal errors on any vote_count value that has decimals.

In [1385]:
# Dtype of release date before conversion.
vote_count_type_before = movies_clean['vote_count'].dtype

# isna count of release date before conversion.
vote_count_missing_before = movies_clean['vote_count'].isna().sum()

# Convert using Int64
movies_clean['vote_count'] = movies_clean['vote_count'].astype('Int64')

# Dtype of release date after conversion.
vote_count_type_after = movies_clean['vote_count'].dtype

# Dtype of release date after conversion.
vote_count_missing_after = movies_clean['vote_count'].isna().sum()

# Print isna count summary.
print(f"vote_count converted to NaN: {vote_count_missing_after - vote_count_missing_before}")

# Print type before and after
print(f"vote_count converted from {vote_count_type_before} to {vote_count_type_after}")

print(f"{(movies_clean['vote_count'] == 0).sum()} movies have no votes")

# Append changes to cleaning log
cleaning_log.append({
    'method': 'conversion',
    'table': 'movies',
    'column': 'vote_count',
    'description': 'float64 to Int64',
    'reason': 'show as whole number',
    'notes': 'no NaN. 44 movies have no votes',
})

vote_count converted to NaN: 0
vote_count converted from float64 to Int64
44 movies have no votes


All vote counts converted without errors. 44 movies have no votes.

### Convert budget to float64

Revenue is already set to float64. We will convert budget to float64 for consistency.

In [1386]:
print(f"Revenue dtype is : {(movies_clean['revenue'].dtype)}")
print(f"Budget dtype before conversion is : {(movies_clean['budget'].dtype)}")

# Dtype of release date after conversion.
budget_type_before = movies_clean['budget'].dtype

# isna count of release date before conversion.
budget_missing_before = movies_clean['budget'].isna().sum()

# Convert using pd.to_datetime
movies_clean['budget'] = movies_clean['budget'].astype('float64')

# Dtype of release date after conversion.
budget_type_after = movies_clean['budget'].dtype

# Dtype of release date after conversion.
budget_missing_after = movies_clean['budget'].isna().sum()

# Print isna count summary.
print(f"budget converted to NaN: {budget_missing_after - budget_missing_before}")

# Print type before and after
print(f"budget converted from {budget_type_before} to {budget_type_after}")

budget_zero_count = (movies_clean['budget'] == 0)
print(f"{budget_zero_count.sum()} movies ({budget_zero_count.mean():.1%}) have 0 budget")

revenue_zero_count = (movies_clean['revenue'] == 0)
print(f"{revenue_zero_count.sum()} movies ({revenue_zero_count.mean():.1%}) have 0 revenue")

# Append changes to cleaning log
cleaning_log.append({
    'method': 'conversion',
    'table': 'movies',
    'column': 'budget',
    'description': 'int64 to float64',
    'reason': 'match revenue dtype',
    'notes': 'no NaN. 48.9% of movies have 0 budget',
})

Revenue dtype is : float64
Budget dtype before conversion is : int64
budget converted to NaN: 0
budget converted from int64 to float64
4443 movies (48.9%) have 0 budget
4322 movies (47.6%) have 0 revenue


Budget was converted with no errors. Each movie has a value in the budget column, but 48.9% are 0. We will filter out movies with missing budget and revenue data from the abt.

### Convert primary_genre, status, and original_language to categories

The columns listed contain few unique values across thousands of rows. Converting them to categories establishes a fixed set of allowed values per column and allows us to create a custom order later if we'd like. ***It also reduces the memory and speed of calculations that use those columns.*** It's not necessary for this project but is good practice when working with large amounts of data.

The memory change shown below shows that converting to category reduced storage by 98%. When working with gigabytes of data this would be useful.

In [1387]:
print("--- Category columns before conversion ---")
# List columns to convert to categories.
category_columns = ['primary_genre', 'status', 'original_language']

print(movies_clean[category_columns].dtypes)

# Column memory usage before converting
category_columns_memory_before = movies_clean[category_columns].memory_usage(deep=True).sum()

# Convert columns to category
movies_clean[category_columns] = movies_clean[category_columns].astype('category')

# Column memory usage after converting
category_columns_memory_after = movies_clean[category_columns].memory_usage(deep=True).sum()

print(f"\nMemory improvement from converting to category: {category_columns_memory_before/1e6:.2f} MB -> {category_columns_memory_after/1e6:.2f} MB")

print("\n--- Unique Values Check ---")
print(f"primary_genre has {movies_clean['primary_genre'].nunique()} unique values.")
print(f"status has {movies_clean['status'].nunique()} unique values.")
print(f"original_language has {movies_clean['original_language'].nunique()} unique values.")

print("\n--- primary_genre Values Check ---")
print(movies_clean["primary_genre"].value_counts().head(3))

print("\n--- status Values Check ---")
print(movies_clean["status"].value_counts().head(3))

# display(movies_clean.query("status == 'Rumored'"))

print("\n--- original_language Values Check ---")
print(movies_clean["original_language"].value_counts().head(3))

# Append changes to cleaning log
cleaning_log.extend([
    {'method': 'conversion', 'table': 'movies', 'column': 'primary_genre', 'description': 'object to category', 'reason': 'few unique values', 'notes': '20 unique values'},
    {'method': 'conversion', 'table': 'movies', 'column': 'status', 'description': 'object to category', 'reason': 'few unique values','notes': '4 unique values'},
    {'method': 'conversion', 'table': 'movies', 'column': 'original_language', 'description': 'object to category', 'reason': 'few unique values','notes': '42 unique values'},
])

--- Category columns before conversion ---
primary_genre        str
status               str
original_language    str
dtype: object

Memory improvement from converting to category: 1.48 MB -> 0.03 MB

--- Unique Values Check ---
primary_genre has 20 unique values.
status has 4 unique values.
original_language has 42 unique values.

--- primary_genre Values Check ---
primary_genre
Drama     2413
Comedy    2168
Action    1177
Name: count, dtype: int64

--- status Values Check ---
status
Released           9061
Rumored              11
Post Production       7
Name: count, dtype: int64

--- original_language Values Check ---
original_language
en    7947
fr     272
ja     185
Name: count, dtype: int64


Converting categorical information to category data type reduced memory usage from 1.48 mb to just .03mb. When tables scale to hundreds of thousands of rows this has a meaningful impact on storage use.

### Movie Conversion Review

Now that we cleaned the movie copy we can compare the raw movies data to the cleaned data and review the change log.

When comparing the raw vs clean data types we can see the 6 columns have been converted and that datatypes have been changed.  

Everything looks accurate. Final adjustments will be made when creating movies_abt.

In [1388]:
# Build the DataFrame
movie_conversion_review = pd.DataFrame({
    'raw_dtype': movies.dtypes, 
    'clean_dtype': movies_clean.dtypes
})

print("--- Raw and clean dtypes ---")
display(movie_conversion_review.transpose())

print("\n--- cleaning log review ---")
display(pd.DataFrame(cleaning_log))

print(f"row count change: {movies.shape[0]:,} -> {movies_clean.shape[0]:,}")
print(f"column count change: {movies.shape[1]} -> {movies_clean.shape[1]}")

--- Raw and clean dtypes ---


,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year
raw_dtype,int64,int64,str,str,str,str,str,float64,int64,float64,float64,float64,float64,str,str,str,bool,str,str,str,int64
clean_dtype,int64,int64,str,str,str,category,datetime64[us],float64,float64,float64,float64,float64,Int64,str,str,str,bool,category,str,category,int64



--- cleaning log review ---


,method,table,column,description,reason,notes
0,conversion,movies,release_date,object to datetime64 (errors=coerce),enable date filtering and pandas date features,"no NaT. min date is 1902-09-01, max date is 20..."
1,conversion,movies,vote_count,float64 to Int64,show as whole number,no NaN. 44 movies have no votes
2,conversion,movies,budget,int64 to float64,match revenue dtype,no NaN. 48.9% of movies have 0 budget
3,conversion,movies,primary_genre,object to category,few unique values,20 unique values
4,conversion,movies,status,object to category,few unique values,4 unique values
5,conversion,movies,original_language,object to category,few unique values,42 unique values


row count change: 9,082 -> 9,082
column count change: 21 -> 21


The first output shows data types from the raw and clean tables as a single dataframe. This makes it easy to compare data types before and after conversion. Row and column counts stay the same since we only converted data.

---
## Aggregation - Ratings

Each row level of the raw ratings table represents a single movie review. The final abt tables will have one row per movie. In order to join them correctly without creating duplicate rows, we must first aggregate the ratings table to transform the row level to equal a single movie.

To transform the row level from a rating to a single movie we will group ratings by movieId then use aggregation to create metrics from the ratings values.

In [ ]:
# Create ratings_aggregated table. 
ratings_aggregated = (
    ratings_clean
    .groupby("movieId", as_index=False)
    .agg(
        rating_count=("rating", "count"),
        avg_rating=("rating", "mean"),
        min_rating=("rating", "min"),
        max_rating=("rating", "max"),
        rating_std_dev = ("rating", "std"),
        distinct_users = ("userId", "nunique"),
        min_rating_date = ("rating_datetime", "min"),
        max_rating_date = ("rating_datetime", "max")
    )
)

ratings_rows_before_agg = ratings_clean.shape[0]
ratings_rows_after_agg = ratings_aggregated.shape[0]
unique_rating_movieids = ratings_clean['movieId'].nunique()
print("--- unique and row count check ---")
print(f"unique movieIds in ratings table: {unique_rating_movieids:,}")
print(f"unique movieIds after aggregation: {ratings_aggregated['movieId'].nunique():,}")
print(f"row count change after aggregation by movieId: {ratings_rows_before_agg:,} -> {ratings_rows_after_agg:,}")

# Verify that 1 review per movie = 1 unique user. Users can't have multiple reviews for a single movie. 
mismatches = ratings_aggregated.query("rating_count != distinct_users").shape[0]
if mismatches != 0:
    print(f"WARNING: {mismatches} movies have duplicate user ratings")
else:
    print("Check passed: each user rated each movie at most once")

print("\n--- isna Check ---")
display(ratings_aggregated.isna().sum())

print("\n--- Aggregation Check ---")
num_rows_with_one_rating = ratings_aggregated.query("rating_count == 1").shape[0]
print(f"{num_rows_with_one_rating:,} movieIds have 1 review. A std dev can't be calculated.")
print(f"{ratings_aggregated.query("rating_count < 20").shape[0]:,} movieIds have <20 reviews.")
print(f"{movies_clean.query("vote_count < 20").shape[0]:,} movieIds from movies table have <20 reviews.")

print("\n--- Describe rating_counts Column ---")
display(ratings_aggregated['rating_count'].describe())

print("\n--- Describe avg_rating Column ---")
display(ratings_aggregated['avg_rating'].describe())

print("\n--- Aggregated Ratings Table Review ---")
display(ratings_aggregated.head(3))

# Log ratings aggregation in cleaning_log
cleaning_log.append({
    'method': 'aggregation',
    'table': 'ratings_aggregated',
    'column': 'rating, userId, rating_datetime',
    'description': 'grouped ratings metrics by movieId',
    'reason': 'transform to 1 row per movie',
    'notes': f"{num_rows_with_one_rating:,} movieIds have 1 review. A std dev can't be calculated.",
})


--- unique and row count check ---
unique movieIds in ratings table: 9,025
unique movieIds after aggregation: 9,025
row count change after aggregation by movieId: 99,810 -> 9,025
Check passed: each user rated each movie at most once

--- isna Check ---


movieId               0
rating_count          0
avg_rating            0
min_rating            0
max_rating            0
rating_std_dev     3042
distinct_users        0
min_rating_date       0
max_rating_date       0
dtype: int64


--- Aggregation Check ---
3,042 movieIds have 1 review. A std dev can't be calculated.
7,725 movieIds have <20 reviews.
1,591 movieIds from movies table have <20 reviews.

--- Describe rating_counts Column ---


count    9025.000000
mean       11.059280
std        24.095433
min         1.000000
25%         1.000000
50%         3.000000
75%         9.000000
max       341.000000
Name: rating_count, dtype: float64


--- Describe avg_rating Column ---


count    9025.000000
mean        3.290203
std         0.881612
min         0.500000
25%         2.833333
50%         3.500000
75%         3.961538
max         5.000000
Name: avg_rating, dtype: float64


--- Aggregated Ratings Table Review ---


,movieId,rating_count,avg_rating,min_rating,max_rating,rating_std_dev,distinct_users,min_rating_date,max_rating_date
0,1,247,3.872470,1.0,5.0,0.958981,247,1996-03-30 19:00:13,2016-10-06 19:55:11
1,2,107,3.401869,1.5,5.0,0.880714,107,1996-03-30 19:12:30,2016-08-01 17:42:33
2,3,59,3.161017,0.5,5.0,1.150115,59,1996-06-05 06:19:04,2016-08-16 22:07:21


Aggregation didn't change the number of keys. Both the ratings and ratings_aggregated tables have 9,025 unique movieIds. It's important to point out that ***3,042 movies only have 1 rating count.*** Standard deviation will be missing for those movies.

### Ratings Threshold

Our initial plan was to create a rating threshold to flag when a movie's average rating is in the top 25%. The 75th percentile of the aggregated rating average is near 4.0. We will use 4 as the rating success threshold.

One thing to note is that avg_ratings with low review counts aren't reliable and should be filtered out during analysis.

In [1414]:
# Assign rating_threshold_met column
ratings_aggregated = ratings_aggregated.assign(rating_threshold_met=lambda x: x["avg_rating"].ge(4).where(x["avg_rating"].notna()).astype("Int64"))

print("ratings flag created!")

print(f"{ratings_aggregated.query('rating_threshold_met == 1').shape[0]:,} movies meet threshold")
print(f"{ratings_aggregated.query('rating_threshold_met == 0').shape[0]:,} movies fail threshold")
print(f"{ratings_aggregated.query('rating_threshold_met == 1').shape[0] + ratings_aggregated.query('rating_threshold_met == 0').shape[0]:,} movies total")

# Log ratings aggregation in cleaning_log
cleaning_log.append({
    'method': 'derivation',
    'table': 'ratings_aggregated',
    'column': 'rating_threshold_met',
    'description': 'created rating success flag',
    'reason': 'can be used for filtering or probability analysis',
    'notes': 'some flags may lack reliable counts',
})


ratings flag created!
2,213 movies meet threshold
6,812 movies fail threshold
9,025 movies total


About 25% of movies meet the ratings threshold.

### Ratings Aggregation Review

Aggregation successfully transformed almost 99,810 rows into 9,025 rows, which matches the number of unique movieIds that exist in the raw ratings table. 

One thing to note is that a large percentage of movies have very few reviews associated with them. The lack of reviews means the avg rating is unreliable for that movie. 3,042 (34%) movieIds have only 1 review and 7,725 (86%) movieIds have fewer than 20 reviews. Compare that with the movies table where only 1,575 movies have a vote count below 20.

Filtering out movies with fewer than 20 reviews in the ratings table leaves us with 1,300 movies to include in the analysis. It's possible that filtering to movies with over 20 reviews introduces selection bias since movies with more reviews are likely more popular and succesful.

In Milestone 3, models will be built using both MovieLens and TMDb ratings to test whether the same movie attributes predict high ratings in each.

---
## Reshaping: Converting Multi-Valued Fields to Long Format

Several columns store multiple values in a single cell, separated by `|`. Splitting and exploding these columns converts them into long format, with one row per movie-value pair. This allows us to analyze the data by individual genre, actor, or keyword. Long data is flexible for grouped summaries and visualizations. For example, we could filter to animated movies and then compare subgenres such as fantasy or comedy.

The following multi-valued columns will be converted to long format:

- `movies['genres_list']`: enables genre-level summaries, such as median ROI by genre.
- `credits['top_cast']`: enables metrics by individual actor, such as median ROI per actor.
- `keywords['keywords_list']`: shows which tags are associated with top-performing movies.

Reshaping creates new tables for exploratory analysis. In these tables, one row represents a single movie-attribute pair (for example, one movie and one genre) rather than a single movie, so each `movieId` or `tmdbId` can appear multiple times. To avoid duplicating movies, these tables are kept separate from the ABT.

In [1391]:
# Create movie_genres table from movies_clean[genres_list]
movie_genres = (
    movies_clean[['movieId', 'genres_list']]
    .assign(genre=movies_clean['genres_list'].str.split('|'))
    .explode('genre')
    .drop(columns='genres_list')
)
movie_genres['genre'] = movie_genres['genre'].str.strip()

display(movie_genres.head(3))
print(f"movie_genres: {movie_genres.shape[0]:,} genres across {movie_genres['movieId'].nunique():,} movies")

# Create movie_cast table from credits[top_cast]
movie_cast = (
    credits_clean[['tmdbId', 'top_cast']]
    .assign(actor=credits_clean['top_cast'].str.split('|'))
    .explode('actor')
    .drop(columns='top_cast')
)
movie_cast['actor'] = movie_cast['actor'].str.strip()

display(movie_cast.head(3))
print(f"movie_cast: {movie_cast.shape[0]:,} actors across {movie_cast['tmdbId'].nunique():,} movies")


# Create movie_keywords table from keywords[keywords_list]
movie_keywords = (
    keywords_clean[['tmdbId', 'keywords_list']]
    .assign(keyword=keywords_clean['keywords_list'].str.split('|'))
    .explode('keyword')
    .drop(columns='keywords_list')
)
movie_keywords['keyword'] = movie_keywords['keyword'].str.strip()

display(movie_keywords.head(3))
print(f"movie_keywords: {movie_keywords.shape[0]:,} keywords across {movie_keywords['tmdbId'].nunique():,} movies")

# Add reshaping to cleaning log
cleaning_log.extend([
    {'method': 'reshaping', 'table': 'movies', 'column': 'genres_list', 'description': 'split on | to create long data', 'reason': 'one row per movie-genre pair', 'notes': f'movie_genres table created kept separate from abt'},
    {'method': 'reshaping', 'table': 'credits', 'column': 'top_cast', 'description': 'split on | to create long data', 'reason': 'one row per movie-actor pair', 'notes': f'movie_cast table created kept separate from abt'},
    {'method': 'reshaping', 'table': 'keywords', 'column': 'keywords_list', 'description': 'split on | to create long data', 'reason': 'one row per movie-keyword pair', 'notes': f'movie_keywords table created kept separate from abt'},
])

,movieId,genre
0,1,Animation
0,1,Comedy
0,1,Family


movie_genres: 22,358 genres across 9,082 movies


,tmdbId,actor
0,862,Tom Hanks
0,862,Tim Allen
0,862,Don Rickles


movie_cast: 26,797 actors across 9,082 movies


,tmdbId,keyword
0,862,jealousy
0,862,toy
0,862,boy


movie_keywords: 47,281 keywords across 9,082 movies


---
## ABT Prep: Final filtering and transormation.

Before we create and save our ABT we need to apply a few more filters and create derived columns. We will create a movies_abt table that is a copy of movies_clean.

In [1392]:
movies_abt = movies_clean
print("movies_abt created!")

movies_abt created!


### Dropping Columns

When creating an abt it's reasonable to drop columns that add no value or are outside the scope of the analysis. 

The movie data contains several such columns:
1) adult: This is a boolean column and all values are false. It provides no value.
2) original_title: For the majority of movies this column is equal to the "title" column. International movies may have the title in their original language.
3) status: This analysis is focused on released movies. We will filter to rows where status is not released then drop the column.
4) production_companies: We will be producing our own original content.
5) production_countires: Many of our movies will be made on our movie lots in the USA.
6) genres: This column is redundant since genres_list has the same info in a better format.
7) popularity: We have decided to use ratings to guage audience score.

In [1393]:
# Save row count before dropping non released status.
row_count_before_status_drop = movies_abt.shape[0]

# filter to where status = Released
movies_abt = movies_abt.query("status == 'Released'")

# Save row count after dropping non released.
row_count_after_status_drop = movies_abt.shape[0]

print(f"After dropping non-released row count went from {row_count_before_status_drop:,} to {row_count_after_status_drop:,}")

# Create a list of columns to drop.
movie_columns_to_drop = ["adult", "original_title", "status", "production_companies"
                        , "production_countries", "genres", "popularity"]

# Column count before dropping columns.
column_count_before_movie_drop = movies_clean.shape[1]

# Drop columns in drop list.
movies_abt = movies_abt.drop(columns= movie_columns_to_drop)

# Column count after dropping columns.
column_count_after_movie_drop = movies_abt.shape[1]

print(f"After dropping columns, column count went from {column_count_before_movie_drop:,} to {column_count_after_movie_drop:,}")

# Append changes to cleaning log
cleaning_log.extend([
    {'method': 'filter', 'table': 'movies_abt', 'column': 'status', 'description': 'filter to status = released', 'reason': 'scope is released movies', 'notes': '21 movies dropped'},
    {'method': 'drop column', 'table': 'movies_abt', 'column': 'status', 'description': 'remove column', 'reason': 'all are released','notes': 'no value added'},
    {'method': 'drop column', 'table': 'movies_abt', 'column': 'original_title', 'description': 'remove column', 'reason': 'will use title','notes': 'redundant'},
    {'method': 'drop column', 'table': 'movies_abt', 'column': 'adult', 'description': 'remove column', 'reason': 'all equal false','notes': 'no value added'},
    {'method': 'drop column', 'table': 'movies_abt', 'column': 'production_companies', 'description': 'remove column', 'reason': 'bringing production in-house','notes': 'no value added'},
    {'method': 'drop column', 'table': 'movies_abt', 'column': 'production_countries', 'description': 'remove column', 'reason': 'movies will be made in USA','notes': 'no value added'},
    {'method': 'drop column', 'table': 'movies_abt', 'column': 'genres', 'description': 'remove column', 'reason': 'prefer genre_list','notes': 'redundant'},
    {'method': 'drop column', 'table': 'movies_abt', 'column': 'popularity', 'description': 'remove column', 'reason': 'out of project scope','notes': 'no value added'},
])

After dropping non-released row count went from 9,082 to 9,061
After dropping columns, column count went from 21 to 14


### Create Derived Columns: ROI and ROI Threshold Flag

The goal for this analysis is to identify which movie attributes are likely to result in a high probability of success. If revenue and budget are available we will use ROI as the measure of success. In the real world, it's common for movies have marketing and other expenses equal to the movie budget. In this analysis we will assume revenue must be double the movie budget to reach profitability.  

We will use assign() to create the roi column and review summary information regarding the new column.

The formula for the derived column roi is: ***roi*** = *(movies[revenue] - movies[budget]) / movies[budget]*

We will also make a roi_threshold_met flag for easy filtering.

The formula for the derived column roi_threshold_met is: ***roi_threshold_met*** = *if roi >=1 then 1 else 0*

In [1394]:
# Assign roi column
movies_abt = movies_abt.assign(roi = lambda x: (x["revenue"] -x["budget"]) / x["budget"] )

# Assign roi_threshold_met column
movies_abt = movies_abt.assign(roi_threshold_met=lambda x: x["roi"].ge(1).where(x["roi"].notna()).astype("Int64"))

# Calculate roi description.
roi_median = movies_abt.query("budget > 0 and revenue > 0")['roi'].median()
roi_count = movies_abt.query("budget > 0 and revenue > 0")['roi'].shape[0]
roi_success = movies_abt.query("roi >= 1 and budget > 0 and revenue > 0")['roi'].shape[0]
roi_success_count = movies_abt.query("roi_threshold_met == 1 and budget > 0 and revenue > 0").shape[0]

print("roi derived column created")

print("\n--- quick validation check ---")
print(f"median roi of movies with budget and rev > 0 is {roi_median:.2f}")
print(f"count of movies with budget and rev > 0 is {roi_count:,} ({roi_count/movies_abt.shape[0]*100:0.0f}% of all movies)  ")
print(f"count of movies with roi >=1: {roi_success:,} ({roi_success/roi_count*100:0.0f}% of all movies with roi calculated)  ")
print(f"count of movies with roi_threshold_met = 1: {roi_success_count:,} ({roi_success_count/roi_count*100:0.0f}% of all movies with roi calculated)  ")

print("\n--- describe() roi data if budget and rev > 0 ---")
display(movies_abt.query("budget > 0 and revenue > 0")['roi'].describe())

print("--- Review isna values that will be removed during filtering ---")
display(movies_abt.isna().sum())

# check isna and isinf
print(f"roi missing: {movies_abt['roi'].isna().sum()}, roi inf: {np.isinf(movies_abt['roi']).sum()}")

# Record roi and roi_threshold_met in cleaning log
cleaning_log.extend([
    {'method': 'derivation', 'table': 'movies_abt', 'column': 'roi', 'description': 'roi = (rev-budget)/budget', 'reason': 'model 1 output: roi >= 1 = successful', 'notes': 'NaN or inf where budget or revenue missing'},
    {'method': 'derivation', 'table': 'movies_abt', 'column': 'roi_threshold_met', 'description': 'if roi >= 1 then 1 else 0', 'reason': 'simple success indicator', 'notes': 'not calculated if roi is na'},
])

roi derived column created

--- quick validation check ---
median roi of movies with budget and rev > 0 is 1.40
count of movies with budget and rev > 0 is 3,846 (42% of all movies)  
count of movies with roi >=1: 2,211 (57% of all movies with roi calculated)  
count of movies with roi_threshold_met = 1: 2,211 (57% of all movies with roi calculated)  

--- describe() roi data if budget and rev > 0 ---


count    3.846000e+03
mean     5.709070e+03
std      2.428872e+05
min     -9.999981e-01
25%      8.610143e-02
50%      1.403340e+00
75%      3.697729e+00
max      1.239638e+07
Name: roi, dtype: float64

--- Review isna values that will be removed during filtering ---


movieId                 0
tmdbId                  0
imdb_id                 0
title                   0
original_language       0
release_date            0
runtime                 0
budget                  0
revenue                 0
vote_average            0
vote_count              0
genres_list            35
primary_genre          35
release_year            0
roi                  3514
roi_threshold_met    3514
dtype: int64

roi missing: 3514, roi inf: 909


***Isna and inf values in the roi columns are expected.*** 

Isna values were created because both the budget and revenue values were missing. Inf values were when roi was calculated on a movie that had 0 budget but revenue was greater than 0. In the next step we will filter out missing budget and revenue values.

### Filter movies_abt by budget and revenue

To calculate roi we need revenue and budget. There are thousands of movies that have 0 for budget and revenue. We will filter out any movies that are missing either number. We'll do this by filtering to movies that have both revenue and budget greater than 10,000. This will get rid of many of the isna values from the prior step.

One thing to note is that some movies may have low revenue because they didn't have a theatrical release or experienced other distribution obstacles. It's possible that these movies would have failed anyway and were removed from a full theatrical release.

In [1395]:
# Create before unique count.
unique_movies_before_filter = movies_abt['movieId'].nunique()

# filter movies_abt to budget and revenue greater than 10,000
movies_abt = movies_abt.query("budget > 10000 & revenue > 10000")

# Create after unique count.
unique_movies_after_filter = movies_abt['movieId'].nunique()

print(f"movieId count change after filtering: {unique_movies_before_filter:,} -> {unique_movies_after_filter:,}")

print("--- describe roi column ---")
display(movies_abt['roi'].describe())

print("--- describe budget column ---")
display(movies_abt['budget'].describe())

print("--- describe revenue column ---")
display(movies_abt['revenue'].describe())

print("--- Review that there are no isna values in budget, revenue, or roi columns ---")
display(movies_abt.isna().sum())

# check isna and isinf
print(f"roi missing: {movies_abt['roi'].isna().sum()}, roi inf: {np.isinf(movies_abt['roi']).sum()}")


print("\n--- review head and top and bottom movies by roi column ---")
display(pd.concat([movies_abt.sort_values(by = 'roi', ascending= False).head(5),
    movies_abt.sort_values(by = 'roi', ascending= False).tail(5)]))

# Record budget and revenue filters in cleaning log
cleaning_log.extend([
    {'method': 'filtering', 'table': 'movies_abt', 'column': 'budget', 'description': 'budget > 10,000', 'reason': 'remove budget outliers and missing budget data', 'notes': f'{unique_movies_before_filter - unique_movies_after_filter:,} movies removed'},
    {'method': 'filtering', 'table': 'movies_abt', 'column': 'revenue', 'description': 'rev > 10,000', 'reason': 'remove rev outliers and missing rev data', 'notes': f'{unique_movies_before_filter - unique_movies_after_filter:,} movies removed'},
])

movieId count change after filtering: 9,061 -> 3,813
--- describe roi column ---


count     3813.000000
mean         9.386864
std        220.317301
min         -0.999486
25%          0.092843
50%          1.407434
75%          3.672060
max      12889.386667
Name: roi, dtype: float64

--- describe budget column ---


count    3.813000e+03
mean     3.701607e+07
std      4.307093e+07
min      1.200000e+04
25%      8.000000e+06
50%      2.200000e+07
75%      5.000000e+07
max      3.800000e+08
Name: budget, dtype: float64

--- describe revenue column ---


count    3.813000e+03
mean     1.129032e+08
std      1.820103e+08
min      1.028900e+04
25%      1.471179e+07
50%      4.738369e+07
75%      1.311835e+08
max      2.787965e+09
Name: revenue, dtype: float64

--- Review that there are no isna values in budget, revenue, or roi columns ---


movieId              0
tmdbId               0
imdb_id              0
title                0
original_language    0
release_date         0
runtime              0
budget               0
revenue              0
vote_average         0
vote_count           0
genres_list          1
primary_genre        1
release_year         0
roi                  0
roi_threshold_met    0
dtype: int64

roi missing: 0, roi inf: 0

--- review head and top and bottom movies by roi column ---


,movieId,tmdbId,imdb_id,title,original_language,release_date,runtime,budget,revenue,vote_average,vote_count,genres_list,primary_genre,release_year,roi,roi_threshold_met
7310,71379,23827,tt1179904,Paranormal Activity,en,2007-09-14,86.0,15000.0,193355800.0,5.9,1351,Horror|Mystery,Horror,2007,12889.386667,1
2164,2710,2667,tt0185937,The Blair Witch Project,en,1999-07-14,81.0,60000.0,248000000.0,6.3,1090,Horror|Mystery,Horror,1999,4132.333333,1
3501,4444,9462,tt0068935,The Way of the Dragon,cn,1972-06-01,100.0,130000.0,85000000.0,7.4,235,Action|Crime,Action,1972,652.846154,1
1867,2361,692,tt0069089,Pink Flamingos,en,1972-03-12,93.0,12000.0,6000000.0,6.2,111,Horror|Comedy|Crime,Horror,1972,499.000000,1
5413,8464,9372,tt0390521,Super Size Me,en,2004-01-17,100.0,65000.0,28575078.0,6.6,520,Documentary|Comedy|Drama,Documentary,2004,438.616585,1
7887,90428,44754,tt0466893,Margaret,en,2011-09-30,149.0,14000000.0,46495.0,6.1,86,Drama,Drama,2011,-0.996679,0
3971,5164,35614,tt0108395,A Troll in Central Park,en,1994-10-07,76.0,23000000.0,71368.0,4.4,40,Fantasy|Animation|Science Fiction|Family,Fantasy,1994,-0.996897,0
8511,109853,242042,tt2355495,Barefoot,en,2014-02-02,90.0,6000000.0,15071.0,6.3,210,Comedy|Drama|Romance,Comedy,2014,-0.997488,0
3268,4092,15785,tt0092746,Cherry 2000,en,1987-11-12,99.0,10000000.0,14000.0,6.0,67,Science Fiction,Science Fiction,1987,-0.998600,0
6474,47254,5289,tt0402910,Chaos,en,2005-01-17,106.0,20000000.0,10289.0,6.1,278,Drama|Action|Crime,Drama,2005,-0.999486,0


***Isna and inf values have been filtered from the data.*** 

### Merge all tables where one row represents a single movie

Merging allows us to combine columns from multiple sources to create a single table. We'll use this function to pull information from the ratings_aggregated, credits, and keywords tables.

Earlier in this project we aggregated the ratings table to convert its row level from one row per review to one row per movie. We can now merge those aggregated metrics with the movies_abt using the matching movieId.

In [1396]:
# Print rows before merge for reference.
print(f"rows before merge: {movies_abt.shape[0]}")

# Merge abt with ratings_aggregated, credits, and keywords tables.
movies_abt = (
    movies_abt
    .merge(ratings_aggregated, on="movieId", how="left", validate="one_to_one")
    .merge(credits, on="tmdbId", how="left", validate="one_to_one")
    .merge(keywords, on="tmdbId", how="left", validate="one_to_one")
)

# Print rows after merge for reference.
print(f"rows after merge: {movies_abt.shape[0]}")

# Check that key column is still unique.
print(f"number of unique movieIds: {movies_abt['movieId'].nunique()}")

# Movies with 1 rating won't have a std dev
print(f"movies with 1 rating: {movies_abt.query("rating_count == 1").shape[0]}")

print("\n--- isna check after merging ---")
display(movies_abt.isna().sum())

movies_abt.head(2)

# Record merge in cleaning log
cleaning_log.append({
    'method': 'merge',
    'table': 'movies_abt, ratings_agg, credits, keywords',
    'column': 'all',
    'description': 'merging to abt',
    'reason': 'combine movie level data',
    'notes': '',
})

rows before merge: 3813
rows after merge: 3813
number of unique movieIds: 3813
movies with 1 rating: 627

--- isna check after merging ---


movieId                   0
tmdbId                    0
imdb_id                   0
title                     0
original_language         0
release_date              0
runtime                   0
budget                    0
revenue                   0
vote_average              0
vote_count                0
genres_list               1
primary_genre             1
release_year              0
roi                       0
roi_threshold_met         0
rating_count             17
avg_rating               17
min_rating               17
max_rating               17
rating_std_dev          644
distinct_users           17
min_rating_date          17
max_rating_date          17
rating_threshold_met     17
director                  0
top_cast                  3
keywords_list            75
dtype: int64

***Important to note that isna values exist in the abt after merging.*** 

There are 2 reasons isna exist after merging:
1) Movies exist in movies_abt that don't exist in the ratings_aggregated, credits, or keywords tables
2) Movies only have 1 rating so the rating standard deviation will appear as NaN.

The math confirms this. 17 movies didn't have a matching movieId in the ratings table. 627 movies only have 1 review, which results in an error for the standard deviation calculation. 

17 + 627 = 644 std dev isna values

***We will leave these movies in the abt for the roi model. They will be removed for ratings model***

### Create Derived Column: Release Month

The abt includes release_date and release_year. By adding release_month we can inspect if releasing movies in a certain month improves the odds of successs. Since we are a streaming service that relies on a steady flow of content to keep subscribers engaged, we may not have to depend on summer blockbusters like traditional production companies. It will still be interesting to explore the different ways month impacts performance.

In [1397]:

print(f"there are {movies_abt['release_date'].isna().sum()} missing release_dates")

# Assign roi column
movies_abt = movies_abt.assign(release_month=lambda x: x["release_date"].dt.month)

print("release_month column created")

print(f"number of isna in release_month column: {movies_abt['release_month'].isna().sum()}")

print("\n--- Unique values check ---")
display(movies_abt.nunique())

print("--- Checking if duplicated titles are valid ---")
display(movies_abt.loc[movies_abt.duplicated(subset='title', keep=False),['movieId','title','release_date']]
    .sort_values(by= ['title','release_date'], ascending = [True, False]).head(10))
    
# Record roi in cleaning log
cleaning_log.append({
    'method': 'derivation',
    'table': 'movies_abt',
    'column': 'release_month',
    'description': 'release_month = month(release_date)',
    'reason': 'can be used for seasonal release analysis',
    'notes': 'no isna values created',
})

there are 0 missing release_dates
release_month column created
number of isna in release_month column: 0

--- Unique values check ---


movieId                 3813
tmdbId                  3813
imdb_id                 3813
title                   3750
original_language         26
release_date            2962
runtime                  144
budget                   459
revenue                 3670
vote_average              56
vote_count              1690
genres_list             1049
primary_genre             18
release_year              93
roi                     3775
roi_threshold_met          2
rating_count             185
avg_rating              1115
min_rating                10
max_rating                10
rating_std_dev          1976
distinct_users           185
min_rating_date         3465
max_rating_date         3649
rating_threshold_met       2
director                1572
top_cast                3738
keywords_list           3689
release_month             12
dtype: int64

--- Checking if duplicated titles are valid ---


,movieId,title,release_date
321,1019,"20,000 Leagues Under the Sea",1954-12-23
2782,62383,"20,000 Leagues Under the Sea",1916-12-24
3015,77798,A Nightmare on Elm Street,2010-04-30
455,1347,A Nightmare on Elm Street,1984-11-14
2988,74789,Alice in Wonderland,2010-03-03
329,1032,Alice in Wonderland,1951-07-03
3156,86293,Arthur,2011-04-08
1230,3524,Arthur,1981-07-17
3660,122888,Ben-Hur,2016-08-17
436,1287,Ben-Hur,1959-12-26


---
## movies_abt Final Review

Our movies_abt table is complete. Let's review what we've done.

We will start with the cleaning log.

In [1398]:
display(pd.DataFrame(cleaning_log))

,method,table,column,description,reason,notes
0,conversion,movies,release_date,object to datetime64 (errors=coerce),enable date filtering and pandas date features,"no NaT. min date is 1902-09-01, max date is 20..."
1,conversion,movies,vote_count,float64 to Int64,show as whole number,no NaN. 44 movies have no votes
2,conversion,movies,budget,int64 to float64,match revenue dtype,no NaN. 48.9% of movies have 0 budget
3,conversion,movies,primary_genre,object to category,few unique values,20 unique values
4,conversion,movies,status,object to category,few unique values,4 unique values
5,conversion,movies,original_language,object to category,few unique values,42 unique values
6,aggregation,ratings_aggregated,"rating, userId, rating_datetime",grouped ratings metrics by movieId,transform to 1 row per movie,"reduced rows from 99,810 to 9,025"
7,derivation,ratings_aggregated,rating_threshold_met,created rating success flag,can be used for filtering or probability analysis,some flags may lack reliable counts
8,reshaping,movies,genres_list,split on | to create long data,one row per movie-genre pair,movie_genres table created kept separate from abt
9,reshaping,credits,top_cast,split on | to create long data,one row per movie-actor pair,movie_cast table created kept separate from abt


***The cleaning log lists every action taken to prepare the data for analysis.*** 

### movies_abt Key Duplicates and Missing Values Check

Our movies_abt table is complete. Let's review what we've done.

We will start with the cleaning log.

In [1399]:
print("--- Duplicated Key Check ---")
display(movies_abt['movieId'].duplicated().sum())

print("--- insa Sums Check ---")
display(movies_abt.isna().sum())

--- Duplicated Key Check ---


np.int64(0)

--- insa Sums Check ---


movieId                   0
tmdbId                    0
imdb_id                   0
title                     0
original_language         0
release_date              0
runtime                   0
budget                    0
revenue                   0
vote_average              0
vote_count                0
genres_list               1
primary_genre             1
release_year              0
roi                       0
roi_threshold_met         0
rating_count             17
avg_rating               17
min_rating               17
max_rating               17
rating_std_dev          644
distinct_users           17
min_rating_date          17
max_rating_date          17
rating_threshold_met     17
director                  0
top_cast                  3
keywords_list            75
release_month             0
dtype: int64

#### Duplicate Keys & Missing Values Review

After creating an abt there are 2 critical checks:
1) Check the table key to ensure there are no duplicates. There are 0, this check passed.
2) Check for missing values. There are missing values, but we acknowledge them and will allow them.
    - 17 unmatched movieIds from the ratings_aggregated table. These movies didn't have a match in the ratings table.
    - 644 missing rating_std_dev values. 627 movies only had 1 rating so a standard deviation could not be calculated. 17 movies don't exist in the aggregated ratings table. These make up the 644 missing values.
    - 75 movies have no keywords_list. These 75 will be excluded from exploratory analysis of the keywords list column.

### Unique Values Check

We will review unique values in each column of the abt. Our main focus is that the primary keys are unique.

In [1415]:
print("\n--- Unique values check ---")
display(movies_abt.nunique())

print("--- Checking if duplicated titles are valid ---")
display(movies_abt.loc[movies_abt.duplicated(subset='title', keep=False),['movieId','title','release_date']]
    .sort_values(by= ['title','release_date'], ascending = [True, False]).head(10))


--- Unique values check ---


movieId                 3813
tmdbId                  3813
imdb_id                 3813
title                   3750
original_language         26
release_date            2962
runtime                  144
budget                   459
revenue                 3670
vote_average              56
vote_count              1690
genres_list             1049
primary_genre             18
release_year              93
roi                     3775
roi_threshold_met          2
rating_count             185
avg_rating              1115
min_rating                10
max_rating                10
rating_std_dev          1976
distinct_users           185
min_rating_date         3465
max_rating_date         3649
rating_threshold_met       2
director                1572
top_cast                3738
keywords_list           3689
release_month             12
dtype: int64

--- Checking if duplicated titles are valid ---


,movieId,title,release_date
321,1019,"20,000 Leagues Under the Sea",1954-12-23
2782,62383,"20,000 Leagues Under the Sea",1916-12-24
3015,77798,A Nightmare on Elm Street,2010-04-30
455,1347,A Nightmare on Elm Street,1984-11-14
2988,74789,Alice in Wonderland,2010-03-03
329,1032,Alice in Wonderland,1951-07-03
3156,86293,Arthur,2011-04-08
1230,3524,Arthur,1981-07-17
3660,122888,Ben-Hur,2016-08-17
436,1287,Ben-Hur,1959-12-26


The check showed that keys are indeed unique. I noticed that there are duplicate movie titles. Upon further inspection we can see that they are mostly remakes or re-releases.

---
### ABT Validation Using SQL

We will use DuckDB to conduct a few high validation checks of the pandas integration steps.

In [1400]:
# query number of distinct movies in movies abt
movie_abt_count = run_duckdb('''
SELECT
    COUNT(DISTINCT(movieid)) as movie_count
FROM movies_abt;
''')

# query number of movies_clean if I apply the budget and revenue filters
filtered_movies_clean_count = run_duckdb('''
SELECT
    COUNT(DISTINCT(movieid)) as movie_count
FROM movies_clean mc
WHERE mc.budget > 10000
    AND mc.revenue >10000
    AND mc.status = 'Released';
''')

# query distinct movieIds from the ratings table.
distinct_rating_movieIds = run_duckdb('''
SELECT
    COUNT(distinct(r.movieId)) as movieId_count
FROM ratings r;
''')

# query the top movie by avg rating and max rate date
top_avg_movie_rating = run_duckdb('''
WITH ratings_aggregated as (
    SELECT
        r.movieId,
        AVG(r.rating) as avg_rating,
        MAX(r.rating_datetime) as max_rating_date
    FROM ratings r
    GROUP BY r.movieId
)
SELECT
    ra.*,
    m.title
FROM ratings_aggregated ra
JOIN movies m on m.movieId = ra.movieId
ORDER BY ra.avg_rating DESC, ra.max_rating_date DESC 
LIMIT 1;
''')

# query number of movies from movies_clean if I apply the budget and revenue filters
count_unmatched_ratings = run_duckdb('''
SELECT
    COUNT(*) as movie_count
FROM movies m
LEFT JOIN ratings r on r.movieId = m.movieId
WHERE m.budget > 10000
    AND m.revenue >10000
    AND m.status = 'Released'
    AND r.movieId IS NULL;
''')

print("--- 1st Check: Unique keys in abt vs raw data ---")
print(f"{movie_abt_count.loc[0,'movie_count']:,} distinct movieIds exist in movies_abt.")
print(f"{filtered_movies_clean_count.loc[0,'movie_count']:,} distinct movieIds in movies_clean if I apply filters from the cleaning_log.")
print("\n--- 2nd Check: Unique keys aggregated vs raw ratings tables---")
print(f"{distinct_rating_movieIds.loc[0,'movieId_count']:,} distinct movieIds exist in the ratings table")
print(f"{ratings_aggregated.shape[0]:,} rows exist in the ratings_aggregated table")
print("\n--- 3rd Check: Validate movies with no ratings ---")
print(f"Joining filtered movies table with ratings using SQL results in {count_unmatched_ratings.loc[0,'movie_count']:,} unmatched movies.")
print(f"movies_abt has {movies_abt['avg_rating'].isna().sum():,} movies with no avg_rating")

print("\n--- 4th Check: Top-rated movie, SQL vs. pandas ---")
print("\n--- top avg rated movie from ratings_aggregated query ---")
display(top_avg_movie_rating)
print("\n--- top avg rated movie from ratings_aggregated table (163949 - they match!) ---")

display(ratings_aggregated.sort_values(by=['avg_rating', 'max_rating_date'] ,ascending= [False,False]) .head(1))

--- 1st Check: Unique keys in abt vs raw data ---
3,813 distinct movieIds exist in movies_abt.
3,813 distinct movieIds in movies_clean if I apply filters from the cleaning_log.

--- 2nd Check: Unique keys aggregated vs raw ratings tables---
9,025 distinct movieIds exist in the ratings table
9,025 rows exist in the ratings_aggregated table

--- 3rd Check: Validate movies with no ratings ---
Joining filtered movies table with ratings using SQL results in 17 unmatched movies.
movies_abt has 17 movies with no avg_rating

--- 4th Check: Top-rated movie, SQL vs. pandas ---

--- top avg rated movie from ratings_aggregated query ---


,movieId,avg_rating,max_rating_date,title
0,163949,5.0,2016-10-14 04:27:19,The Beatles: Eight Days a Week - The Touring Y...



--- top avg rated movie from ratings_aggregated table (163949 - they match!) ---


,movieId,rating_count,avg_rating,min_rating,max_rating,rating_std_dev,distinct_users,min_rating_date,max_rating_date,rating_threshold_met
9024,163949,1,5.0,5.0,5.0,NaN,1,2016-10-14 04:27:19,2016-10-14 04:27:19,1


#### SQL Validaiton Review

Using SQL, I was able to run some high level validation checks:
1) The first check compares unique keys in movies_abt with the unique keys in the raw movies table after applying the filters listed in the cleaning_log. They both equal 3,813.
2) The second check compares distinct movieIds in the raw ratings table with the numver of rows in the aggregated ratings table. They both show 9,025 movies.
3) The third check uses a left join where null to confirm that there are 17 movies with no match in the ratings table.
4) The fourth check uses 2 different methods to find the same result. Aggregating the raw data using a CTE and sorting results in the same movie if I apply the same sorting to the ratings_aggregated table.

---
## Save Processed Tables

Now that we've documented all the changes and verified the outputs, we will save all dataframes created throughout the process.

In [1401]:
output_files = {
    'movies_abt': movies_abt,
    'movies_clean': movies_clean,
    'ratings_aggregated': ratings_aggregated,
    'movie_genres': movie_genres,
    'movie_cast': movie_cast,
    'movie_keywords': movie_keywords,
    'cleaning_log': pd.DataFrame(cleaning_log),
}

for name, df in output_files.items():
    path = PROCESSED_DATA / f"{name}.csv"
    df.to_csv(path, index=False)
    print(f"Saved {len(df):,} rows to {path}")

Saved 3,813 rows to c:\Users\matth\Project-Option-1-Matt-Flores\outputs\processed\movies_abt.csv
Saved 9,082 rows to c:\Users\matth\Project-Option-1-Matt-Flores\outputs\processed\movies_clean.csv
Saved 9,025 rows to c:\Users\matth\Project-Option-1-Matt-Flores\outputs\processed\ratings_aggregated.csv
Saved 22,358 rows to c:\Users\matth\Project-Option-1-Matt-Flores\outputs\processed\movie_genres.csv
Saved 26,797 rows to c:\Users\matth\Project-Option-1-Matt-Flores\outputs\processed\movie_cast.csv
Saved 47,281 rows to c:\Users\matth\Project-Option-1-Matt-Flores\outputs\processed\movie_keywords.csv
Saved 25 rows to c:\Users\matth\Project-Option-1-Matt-Flores\outputs\processed\cleaning_log.csv


---
## AI Use Disclosure

I used Claude as a tutor and reviewer throughout this milestone. In line with the course AI guidelines, AI was used as a reference and coding assistant, not to complete the analysis for me.

**How AI was used:**
- **Concept explanations:** e.g., aggregation and row meaning, long vs. wide data, data leakage, `size` vs. `count`, why floats allow `NaN`, nullable `Int64`, and DuckDB.
- **Syntax help and debugging:** pandas methods (`.dt.month`, `.loc`, `sort_values`, `explode`, `merge(validate=...)`, f-string formatting) and DuckDB query structure. AI helped identify bugs, including a table copied from the wrong source, `inf` ROI values being flagged as successes, and a revenue threshold mismatch found through a SQL check.
- **Code examples:** some short code patterns were suggested by AI (e.g., the target flag pattern using `.ge().where()`.
- **Writing feedback:** AI reviewed my markdown for clarity, accuracy, and grammar.
- **Background research:** context on specific films (e.g., limited theatrical releases) and Netflix release strategy.

**My own work:** I made all analytical decisions, including the business question, the two-model design, the success thresholds (ROI ≥ 1, average rating ≥ 4.0), the $10,000 budget and revenue filters, which columns to drop, and the single-ABT structure. I wrote the majority of the code and documentation, and I can explain every step in this notebook.